# 3. Step 3: Model Creation, Hyperparameter Search, and Model Evaluation #

## Goals: ##
* Create train test splits and implement a RandomForestClassifier or a GradientBoostingClassifier 
* Hyperparameter tuning via GridSearchCV or RandomizedSearchCV
* Upon finding optimal hyperparameters, we re-train model using these hyperparameters, generate predictions for this new model, and output the subsequent F1 score for this classifier. 

## Import packages ##

In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    f1_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)
from scipy.stats import randint, uniform

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ModuleNotFoundError:
    XGB_AVAILABLE = False

## Load cleaned data ##

In [ ]:
df_all_types = pd.read_csv("../data/cleaned_all_types_bank_transactions.csv")
df_only_fraud_types = pd.read_csv("../data/cleaned_only_fraud_types_bank_transactions.csv")


In [10]:
df_all_types.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,hour,diff_orig,diff_dest,amt_equals_orig_balance,orig_balance_zero_after,dest_balance_unchanged,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,1,9839.64,0.0,0,0,1,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,1,1864.28,0.0,0,0,1,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,1,181.00,0.0,1,1,1,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,1,181.00,-21182.0,1,1,0,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,1,11668.14,0.0,0,0,1,False,False,True,False


In [11]:
df_only_fraud_types.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,hour,diff_orig,diff_dest,amt_equals_orig_balance,orig_balance_zero_after,dest_balance_unchanged
0,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1,1,181.0,0.00,1,1,1
1,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1,1,181.0,-21182.00,1,1,0
2,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0,1,15325.0,46430.44,0,1,0
3,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0,1,705.0,-22425.00,0,1,0
4,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0,1,10835.0,2712905.89,0,1,0


In [ ]:
X_all_types = df_all_types.drop(columns=["isFraud"])
y_all_types = df_all_types["isFraud"]

X_fraud_types_only = df_only_fraud_types.drop(columns=["isFraud"])
y_fraud_types_only = df_only_fraud_types["isFraud"]

## Train/test split ##

In [13]:
X_all_types_train, X_all_types_test, y_all_types_train, y_all_types_test = train_test_split(
    X_all_types, y_all_types, test_size=0.2, random_state=42, stratify=y_all_types
)

X_fraud_types_only_train, X_fraud_types_only_test, y_fraud_types_only_train, y_fraud_types_only_test = train_test_split(
    X_fraud_types_only, y_fraud_types_only, test_size=0.2, random_state=42, stratify=y_fraud_types_only
)

## Additional Preprocessing ##

Although data cleaning and transformation were completed in Step 2, the modeling step revealed that additional preprocessing was still needed to make the data fully compatible with the models. When training the models, it became clear that the dataset still contained categorical variables that needed to be encoded and numeric variables that benefited from scaling inside the modeling pipeline. This step was therefore moved into the modeling stage so that the transformations could be applied consistently to both the training and test sets, avoid data leakage, and ensure that each model received input in the format it required.


In [35]:
num_cols_all = X_all_types.select_dtypes(include=np.number).columns
cat_cols_all = X_all_types.select_dtypes(exclude=np.number).columns

num_cols_fraud = X_fraud_types_only.select_dtypes(include=np.number).columns
cat_cols_fraud = X_fraud_types_only.select_dtypes(exclude=np.number).columns

preprocess_all = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_all),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_all),
    ]
)

preprocess_fraud = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_fraud),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_fraud),
    ]
)


## Hyperparameter grids ##

The hyperparameter grids were defined separately from the model training code so that model could be tuned in a clear and flexible way. Because hyperparameters are set before training rather than learned from the data, they must be specified in advance and passed into the search procedure as a set of candidate values. Defining these grids separately also makes it easier to reuse the same tuning logic across different models, adjust the search space when needed, and keep the modeling code organized and readable.


In [ ]:
lr_param_dist = {
    "logreg__C": uniform(0.01, 10),
    "logreg__solver": ["lbfgs"],
    "logreg__max_iter": [2000],
}

rf_param_dist = {
    "rf__n_estimators": randint(25, 75),
    "rf__max_depth": [None, 10, 15],
    "rf__min_samples_split": randint(2, 6),
    "rf__min_samples_leaf": randint(1, 4),
    "rf__max_features": ["sqrt"],
}

xgb_param_dist = {
    "xgb__n_estimators": [50, 100],
    "xgb__max_depth": [3, 5],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__subsample": [0.8],
    "xgb__colsample_bytree": [0.8],
}


## Helper function ##

To keep the modeling workflow organized and reusable, a few helper functions were created to handle repeated tasks such as extracting prediction scores and training, tuning, and evaluating each model. This approach makes the notebook easier to follow because the same evaluation steps can be applied consistently across Logistic Regression, Random Forest, and XGBoost without rewriting the same code multiple times. It also improves readability and makes later debugging or updates easier, since any change to the evaluation logic only needs to be made in one place.


In [38]:
def get_scores(model, X_test):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_test)[:, 1]
    return model.decision_function(X_test)

def tune_train_eval(model, param_dist, X_train, y_train, X_test, y_test, model_name, n_iter=5):
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="f1",
        cv=2,
        random_state=42,
        n_jobs=1,
        verbose=0,
        pre_dispatch="2*n_jobs",
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_score = get_scores(best_model, X_test)
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc = roc_auc_score(y_test, y_score)

    print("\n" + "=" * 70)
    print(model_name)
    print("Best parameters:", search.best_params_)
    print(f"Test F1 score: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Test ROC AUC:   {auc:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return best_model, search, fpr, tpr, auc

## Logistic Regression baseline on all transaction types ##

In [39]:
lr_all = Pipeline([
    ("preprocess", preprocess_all),
    ("logreg", LogisticRegression(random_state=42, class_weight="balanced"))
])

lr_all, lr_all_search, lr_all_fpr, lr_all_tpr, lr_all_auc = tune_train_eval(
    lr_all,
    lr_param_dist,
    X_all_types_train, y_all_types_train,
    X_all_types_test, y_all_types_test,
    "Logistic Regression - All Types",
    n_iter=4
)


Logistic Regression - All Types
Best parameters: {'logreg__C': np.float64(3.7554011884736247), 'logreg__max_iter': 2000, 'logreg__solver': 'lbfgs'}
Test F1 score: 0.9753
Test ROC AUC:   1.0000
Confusion matrix:
[[1270802      79]
 [      4    1639]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.95      1.00      0.98      1643

    accuracy                           1.00   1272524
   macro avg       0.98      1.00      0.99   1272524
weighted avg       1.00      1.00      1.00   1272524



The Logistic Regression baseline performed surprisingly well, with an AUC of 1.0 and an F1 score of 0.9753. That suggests the features already do a very good job of separating fraudulent from non-fraudulent transactions, so the classes may be fairly easy to distinguish in this feature space. At the same time, the result is strong enough that it’s worth being cautious: the dataset is heavily imbalanced, which can make performance look better than it really is, and there’s also a chance that some features are unusually informative or that leakage is inflating the scores.


## Random Forest on all transaction types ##


In [40]:
rf_all = Pipeline([
    ("preprocess", preprocess_all),
    ("rf", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
        n_estimators=50,
        max_depth=10
    ))
])

rf_all, rf_all_search, rf_all_fpr, rf_all_tpr, rf_all_auc = tune_train_eval(
    rf_all,
    rf_param_dist,
    X_all_types_train, y_all_types_train,
    X_all_types_test, y_all_types_test,
    "Random Forest - All Types",
    n_iter=3
)



Random Forest - All Types
Best parameters: {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 4, 'rf__n_estimators': 43}
Test F1 score: 0.9988
Test ROC AUC:   0.9988
Confusion matrix:
[[1270881       0]
 [      4    1639]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524



Random Forest also performed extremely well on the all-types dataset and only improved slightly over the already strong Logistic Regression baseline. It classified almost everything correctly, missing only four fraud cases and producing no false positives, which suggests that the data contains very strong signals that make fraud relatively easy to identify. Since the baseline model was already performing at a very high level, this result seems to reflect how clearly the classes are separated in the data rather than a dramatic improvement from the Random Forest model itself.

## XGBoost on all transaction types ##

In [41]:
if XGB_AVAILABLE:
    scale_pos_weight_all = (y_all_types_train == 0).sum() / (y_all_types_train == 1).sum()

    xgb_all = Pipeline([
        ("preprocess", preprocess_all),
        ("xgb", XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight_all
        ))
    ])

    xgb_all, xgb_all_search, xgb_all_fpr, xgb_all_tpr, xgb_all_auc = tune_train_eval(
        xgb_all,
        xgb_param_dist,
        X_all_types_train, y_all_types_train,
        X_all_types_test, y_all_types_test,
        "XGBoost - All Types",
        n_iter=5
    )
else:
    xgb_all = None
    xgb_all_search = None
    xgb_all_fpr = xgb_all_tpr = xgb_all_auc = None
    print("\n" + "=" * 70)
    print("XGBoost - All Types")
    print("XGBoost is not installed, so this model was skipped.")


XGBoost - All Types
Best parameters: {'xgb__subsample': 0.8, 'xgb__n_estimators': 50, 'xgb__max_depth': 5, 'xgb__learning_rate': 0.05, 'xgb__colsample_bytree': 0.8}
Test F1 score: 0.9982
Test ROC AUC:   0.9999
Confusion matrix:
[[1270879       2]
 [      4    1639]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524



Like the previous two models, XGBoost also performed extremely well on the all-types dataset, with results that were very close to perfect. It missed only four fraud cases and produced just two false positives, which shows that the model was able to separate the two classes very effectively. Compared with Random Forest, the improvement is very small, which probably means that the dataset itself is doing most of the work here. The fraud and non-fraud classes are already very clearly separated, so even different model types end up performing at nearly the same level.


## Logistic Regression baseline on fraud-only types ##


In [42]:
lr_fraud = Pipeline([
    ("preprocess", preprocess_fraud),
    ("logreg", LogisticRegression(random_state=42, class_weight="balanced"))
])

lr_fraud, lr_fraud_search, lr_fraud_fpr, lr_fraud_tpr, lr_fraud_auc = tune_train_eval(
    lr_fraud,
    lr_param_dist,
    X_fraud_types_only_train, y_fraud_types_only_train,
    X_fraud_types_only_test, y_fraud_types_only_test,
    "Logistic Regression - TRANSFER and CASH_OUT Only",
    n_iter=4
)


Logistic Regression - TRANSFER and CASH_OUT Only
Best parameters: {'logreg__C': np.float64(7.3299394181140505), 'logreg__max_iter': 2000, 'logreg__solver': 'lbfgs'}
Test F1 score: 0.9906
Test ROC AUC:   0.9996
Confusion matrix:
[[552415     24]
 [     7   1636]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.99      1.00      0.99      1643

    accuracy                           1.00    554082
   macro avg       0.99      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082



Compared with the earlier all-types models, the fraud-only Logistic Regression performed very strongly, but it was slightly below the Random Forest and XGBoost results and a bit different from the all-types Logistic Regression baseline. It still achieved near-perfect performance, which shows that narrowing the dataset to `TRANSFER` and `CASH_OUT` made the fraud signal even clearer. At the same time, because all of the models are performing so well, the differences between them are relatively small, so the main takeaway is that the dataset itself is highly predictive rather than that one model is dramatically outperforming the others.

## Random Forest on only fraud-bearing transaction types ##


In [ ]:
rf_fraud = Pipeline([
    ("preprocess", preprocess_fraud),
    ("rf", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
        n_estimators=50,
        max_depth=10
    ))
])

rf_fraud, rf_fraud_search, rf_fraud_fpr, rf_fraud_tpr, rf_fraud_auc = tune_train_eval(
    rf_fraud,
    rf_param_dist,
    X_fraud_types_only_train, y_fraud_types_only_train,
    X_fraud_types_only_test, y_fraud_types_only_test,
    "Random Forest - TRANSFER and CASH_OUT Only",
    n_iter=3
)


Random Forest - All Types
Best parameters: {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 4, 'rf__n_estimators': 43}
Test F1 score: 0.9988
Test ROC AUC:   0.9988
Confusion matrix:
[[1270881       0]
 [      4    1639]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524



Random Forest performed almost identically to Logistic Regression on the all-types dataset, with only a slight improvement. It missed just four fraud cases and made no false positives, which reinforces the earlier takeaway that the classes are already very easy to separate in this feature space.


## XGBoost on only fraud-bearing transaction types  ##

In [ ]:
if XGB_AVAILABLE:
    scale_pos_weight_fraud = (y_fraud_types_only_train == 0).sum() / (y_fraud_types_only_train == 1).sum()

    xgb_fraud = Pipeline([
        ("preprocess", preprocess_fraud),
        ("xgb", XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight_fraud
        ))
    ])

    xgb_fraud, xgb_fraud_search, xgb_fraud_fpr, xgb_fraud_tpr, xgb_fraud_auc = tune_train_eval(
        xgb_fraud,
        xgb_param_dist,
        X_fraud_types_only_train, y_fraud_types_only_train,
        X_fraud_types_only_test, y_fraud_types_only_test,
        "XGBoost - TRANSFER and CASH_OUT Only",
        n_iter=5
    )
else:
    xgb_fraud = None
    xgb_fraud_search = None
    xgb_fraud_fpr = xgb_fraud_tpr = xgb_fraud_auc = None
    print("\n" + "=" * 70)
    print("XGBoost - TRANSFER and CASH_OUT Only")
    print("XGBoost is not installed, so this model was skipped.")


XGBoost - TRANSFER and CASH_OUT Only
Best parameters: {'xgb__subsample': 0.8, 'xgb__n_estimators': 50, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.05, 'xgb__colsample_bytree': 0.8}
Test F1 score: 0.9960
Test ROC AUC:   0.9993
Confusion matrix:
[[552432      7]
 [     6   1637]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       1.00      1.00      1.00      1643

    accuracy                           1.00    554082
   macro avg       1.00      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082



XGBoost performed very strongly on the fraud-only dataset, but its results were still in the same general range as the other models. It missed only a few fraud cases and produced a small number of false positives, which shows that the model was able to capture the fraud pattern well without any major difficulty. The main takeaway is that narrowing the data to `TRANSFER` and `CASH_OUT` makes the signal very clear, so XGBoost performs well but does not dramatically change the overall performance.
